## WaveNet Neural Network for EEG Data Classification


In [1]:
# Importing necessities 
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, Add, Activation, Multiply
from tensorflow.keras.models import Model
import numpy as np
import matplotlib.pyplot as plt

from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt # plotting
import numpy as np # linear algebra
import os # accessing directory structure
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import csv
import shutil
import pathlib 
import copy
import random
import itertools


import matplotlib.pyplot as plt
import pandas as pd 
import numpy as np
import scipy.signal as signal
import scipy.stats as stats
import scipy.io as sio
import matplotlib.pyplot as plt


from random import sample as sm

import glob
import os
import mat73

import os
import h5py
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure

import shutil
from sklearn.model_selection import train_test_split

import os
import numpy as np
from scipy.io import loadmat

import tqdm
from tqdm import tqdm
import time


## Create HDF5 file

NameError: name 'time' is not defined

## Split data

In [2]:
dataset_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project'
output_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DataSetTest\Output'
start_time = time.time()

# Define the directory containing your dataset
#dir = os.path.join(dataset_dir, 'DatasetFnusa\\DatasetFnusa')
#dir = os.path.join(dataset_dir, 'DatasetMayo\\DatasetMayo')
#dir = os.path.join(dataset_dir, 'DataSetTest') # Used to test pipeline 
dir = os.path.join(dataset_dir, 'Dataset_Joined') #ataset Fnusa and Mayo together. Easier to get through the pipeline
#dir = os.path.join(dataset_dir, 'NEWTEST') # Used to double check wavenetmodel after first test

# Define the output HDF5 file in order to minimize memory usage
output_file = os.path.join(output_dir, 'dataset.h5')

import h5py
import numpy as np
import os

def process_file(file, label, hdf5_file):
if file.endswith('.mat'):
data_path = os.path.join(dir, str(label), file)
try:
mat_file = mat73.loadmat(data_path)
data = mat_file['data'].T
except:
mat_file = sio.loadmat(data_path)
data = mat_file['data'].flatten()
hdf5_file.create_dataset(f'label_{label}/file_{file}', data=data)

def split_and_save_data(file_path, output_dir, labels, train_ratio=0.0, val_ratio=0.0, test_ratio=1):
    with h5py.File(file_path, 'r') as hdf5_file:
        for label in labels:
            label_group = hdf5_file[f'label_{label}']
            file_names = list(label_group.keys())
            np.random.shuffle(file_names)
            
            num_samples = len(file_names)
            num_train = int(train_ratio * num_samples)
            num_val = int(val_ratio * num_samples)
            
            train_files = file_names[:num_train]
            val_files = file_names[num_train:num_train + num_val]
            test_files = file_names[num_train + num_val:]

            save_to_hdf5(output_dir, f'train_label_{label}.h5', label_group, train_files)
            save_to_hdf5(output_dir, f'val_label_{label}.h5', label_group, val_files)
            save_to_hdf5(output_dir, f'test_label_{label}.h5', label_group, test_files)

def save_to_hdf5(output_dir, file_name, label_group, file_names):
    with h5py.File(os.path.join(output_dir, file_name), 'w') as hdf5_file:
        group = hdf5_file.create_group(f'label')
        for file_name in file_names:
            data = label_group[file_name][:]
            group.create_dataset(file_name, data=data)

split_and_save_data(output_file, output_dir, labels)



end_time = time.time()
elapsed_time = end_time - start_time
print("Elapsed time:", elapsed_time, "seconds")

IndentationError: expected an indented block (3157517292.py, line 20)

## Create TF dataset in batches & split it

In [3]:
dataset_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project'
output_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DataSetTest\Output'
start_time = time.time()

# Define the directory containing your dataset
#dir = os.path.join(dataset_dir, 'DatasetFnusa\\DatasetFnusa')
#dir = os.path.join(dataset_dir, 'DatasetMayo\\DatasetMayo')
#dir = os.path.join(dataset_dir, 'DataSetTest') # Used to test pipeline 
dir = os.path.join(dataset_dir, 'DataSetTest') #ataset Fnusa and Mayo together. Easier to get through the pipeline

# Define the output HDF5 file in order to minimize memory usage
output_file = os.path.join(output_dir, 'dataset.h5')
file_path=output_dir

# Function to count the number of samples in the HDF5 file
def count_samples(file_path, labels):
    sample_counts = {label: 0 for label in labels}
    with h5py.File(file_path, 'r') as hdf5_file:
        for label in labels:
            label_group = hdf5_file[f'label_{label}']
            sample_counts[label] = len(label_group.keys())
    return sample_counts


# Generator function to read data from HDF5 file in batches
def hdf5_generator(file_path, batch_size):
    with h5py.File(file_path, 'r') as hdf5_file:
        label_group = hdf5_file['label']
        file_names = list(label_group.keys())
        np.random.shuffle(file_names)
        for start_idx in range(0, len(file_names), batch_size):
            batch_data = []
            batch_labels = []
            end_idx = min(start_idx + batch_size, len(file_names))
            for i in range(start_idx, end_idx):
                file_name = file_names[i]
                data = label_group[file_name][:]
                batch_data.append(data)
                batch_labels.append(int(file_path.split('_')[-1].split('.')[0]))  # Extract label from filename
            batch_data = np.array(batch_data)
            batch_labels = np.array(batch_labels)
            yield batch_data.reshape((batch_data.shape[0], 1, 15000)), batch_labels.reshape((batch_labels.shape[0], 1))

# Function to create TensorFlow dataset from generator
def create_tf_dataset_from_generator(file_paths, batch_size):
    datasets = []
    for file_path in file_paths:
        dataset = tf.data.Dataset.from_generator(
            lambda: hdf5_generator(file_path, batch_size),
            output_signature=(
                tf.TensorSpec(shape=(None, 1, 15000), dtype=tf.float64),
                tf.TensorSpec(shape=(None, 1), dtype=tf.int64)
            )
        )
        datasets.append(dataset)
    combined_dataset = tf.data.Dataset.sample_from_datasets(datasets, weights=[1/len(datasets)]*len(datasets))
    return combined_dataset.repeat()  # Ensure the dataset can generate enough batches

# Define file path and labels

labels = [0, 1, 2, 3]  # The labels in your dataset

# Count the number of samples for each label
sample_counts = count_samples(output_file, labels)
total_samples = sum(sample_counts.values())

# Determine a reasonable batch size based on the total number of samples
max_batch_size = 128  # Adjust as necessary
min_batch_size = 32   # Adjust as necessary
batch_size = max(min(total_samples // 10, max_batch_size), min_batch_size)

# Create TensorFlow datasets
train_files = [os.path.join(output_dir, f'train_label_{label}.h5') for label in labels]
val_files = [os.path.join(output_dir, f'val_label_{label}.h5') for label in labels]
test_files = [os.path.join(output_dir, f'test_label_{label}.h5') for label in labels]

train_dataset = create_tf_dataset_from_generator(train_files, batch_size)
val_dataset = create_tf_dataset_from_generator(val_files, batch_size)
test_dataset = create_tf_dataset_from_generator(test_files, batch_size)

# Verify the splits by checking the first batch
for data_batch, label_batch in train_dataset.take(1):
    print(f"Training data batch shape: {data_batch.shape}")
    print(f"Training labels batch shape: {label_batch.shape}")
for data_batch, label_batch in val_dataset.take(1):
    print(f"Validation data batch shape: {data_batch.shape}")
    print(f"Validation labels batch shape: {label_batch.shape}")
for data_batch, label_batch in test_dataset.take(1):
    print(f"Test data batch shape: {data_batch.shape}")
    print(f"Test labels batch shape: {label_batch.shape}")
print("Sample count:", sample_counts)
print("Total samples:", total_samples)
end_time = time.time()
elapsed_time = end_time - start_time
print("Elapsed time:", elapsed_time, "seconds")


Training data batch shape: (57, 1, 15000)
Training labels batch shape: (57, 1)
Validation data batch shape: (13, 1, 15000)
Validation labels batch shape: (13, 1)
Test data batch shape: (14, 1, 15000)
Test labels batch shape: (14, 1)
Sample count: {0: 101, 1: 91, 2: 252, 3: 131}
Total samples: 575
Elapsed time: 0.9842135906219482 seconds


In [86]:
import os
import h5py

# Function to get all file names from the HDF5 file for a specific label
def get_filenames(file_path):
    filenames = set()
    with h5py.File(file_path, 'r') as hdf5_file:
        label_group = hdf5_file['label']
        filenames.update(label_group.keys())
    return filenames

# Function to check for overlap between datasets
def check_for_overlap(train_files, val_files, test_files):
    train_filenames = set()
    val_filenames = set()
    test_filenames = set()

    # Collect filenames from all training files
    for file in train_files:
        train_filenames.update(get_filenames(file))
    
    # Collect filenames from all validation files
    for file in val_files:
        val_filenames.update(get_filenames(file))
    
    # Collect filenames from all test files
    for file in test_files:
        test_filenames.update(get_filenames(file))
    
    # Check for overlaps
    train_val_overlap = train_filenames.intersection(val_filenames)
    train_test_overlap = train_filenames.intersection(test_filenames)
    val_test_overlap = val_filenames.intersection(test_filenames)
    
    print(f"Number of overlapping files between train and val: {len(train_val_overlap)}")
    print(f"Number of overlapping files between train and test: {len(train_test_overlap)}")
    print(f"Number of overlapping files between val and test: {len(val_test_overlap)}")

    return train_val_overlap, train_test_overlap, val_test_overlap

# Define file paths for each dataset
train_files = [os.path.join(output_dir, f'train_label_{label}.h5') for label in labels]
val_files = [os.path.join(output_dir, f'val_label_{label}.h5') for label in labels]
test_files = [os.path.join(output_dir, f'test_label_{label}.h5') for label in labels]

# Check for overlaps
train_val_overlap, train_test_overlap, val_test_overlap = check_for_overlap(train_files, val_files, test_files)

# Output overlap details if any
if train_val_overlap:
    print("Overlap between train and val datasets:")
    for file in train_val_overlap:
        print(file)

if train_test_overlap:
    print("Overlap between train and test datasets:")
    for file in train_test_overlap:
        print(file)

if val_test_overlap:
    print("Overlap between val and test datasets:")
    for file in val_test_overlap:
        print(file)


Number of overlapping files between train and val: 0
Number of overlapping files between train and test: 0
Number of overlapping files between val and test: 0


## Compile and train the model

In [4]:
# Define and compile the model
input_shape = (1, 15000)
wavenet_model = tf.keras.Sequential()
wavenet_model.add(tf.keras.layers.Input(shape=input_shape))

# Define Swish activation function
def swish_activation(x):
    return x * tf.sigmoid(x)

# Add dropout regularization
wavenet_model.add(tf.keras.layers.BatchNormalization())
wavenet_model.add(tf.keras.layers.Dropout(0.2))

for rate in [1, 2, 4, 8, 16, 32, 64, 128]:
    wavenet_model.add(tf.keras.layers.Conv1D(
        filters=32, kernel_size=2, padding="causal", activation=swish_activation,
        dilation_rate=rate))
    wavenet_model.add(tf.keras.layers.Dropout(0.2))

wavenet_model.add(tf.keras.layers.Conv1D(filters=14, kernel_size=1))

# Print the model summary
wavenet_model.summary()

# Compile the model
wavenet_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Define early stopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',  # Monitor the validation loss
    patience=1,          # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True  # Restore model weights from the epoch with the best value of the monitored quantity
)

# Train the model with early stopping
history = wavenet_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=[early_stopping]  # Add early stopping callback
)

# Evaluate the model
test_loss, test_accuracy = wavenet_model.evaluate(test_dataset, steps=test_steps)
print(f"Test accuracy: {test_accuracy}")



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 batch_normalization (BatchN  (None, 1, 15000)         60000     
 ormalization)                                                   
                                                                 
 dropout (Dropout)           (None, 1, 15000)          0         
                                                                 
 conv1d (Conv1D)             (None, 1, 32)             960032    
                                                                 
 dropout_1 (Dropout)         (None, 1, 32)             0         
                                                                 
 conv1d_1 (Conv1D)           (None, 1, 32)             2080      
                                                                 
 dropout_2 (Dropout)         (None, 1, 32)             0         
                                                        

NameError: name 'steps_per_epoch' is not defined

In [6]:
def count_samples_model(file_path):
    total_samples = 0
    with h5py.File(file_path, 'r') as hdf5_file:
        # Access the 'label' group
        label_group = hdf5_file['label']
        total_samples = len(label_group.keys())
    return total_samples

In [2]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, Callback
# Define the directories containing your train, validation, and test files
#dataset_dir = 
#output_dir =
#dir = os.path.join(dataset_dir, 'DatasetTest')
start_time = time.time()
output_file = os.path.join(output_dir, 'dataset.h5')
train_files = [os.path.join(output_dir, f'train_label_{label}.h5') for label in labels]
val_files = [os.path.join(output_dir, f'val_label_{label}.h5') for label in labels]
test_files = [os.path.join(output_dir, f'test_label_{label}.h5') for label in labels]

# Calculate total samples for each dataset
total_train_samples = sum(count_samples_model(file) for file in train_files)
total_val_samples = sum(count_samples_model(file) for file in val_files)
total_test_samples = sum(count_samples_model(file) for file in test_files)

# Calculate steps per epoch
steps_per_epoch = int(total_train_samples // batch_size)
validation_steps = int(total_val_samples // batch_size)
test_steps = int(total_test_samples // batch_size)

# Ensure the steps are at least 1
steps_per_epoch = max(steps_per_epoch, 1)
validation_steps = max(validation_steps, 1)
test_steps = max(test_steps, 1)

# Define Swish activation function
def swish_activation(x):
    return x * tf.sigmoid(x)

class StepHistory(Callback):
    def on_train_begin(self, logs={}):
        self.step_losses = []
        self.step_acc = []
        self.val_step_losses = []
        self.val_step_acc = []

    def on_batch_end(self, batch, logs={}):
        self.step_losses.append(logs.get('loss'))
        self.step_acc.append(logs.get('accuracy'))

    def on_epoch_end(self, epoch, logs={}):
        self.val_step_losses.append(logs.get('val_loss'))
        self.val_step_acc.append(logs.get('val_accuracy'))
        
step_history = StepHistory()

# Build and compile the WaveNet model
wavenet_model = tf.keras.Sequential()

# Add dropout regularization
wavenet_model.add(tf.keras.layers.BatchNormalization())
wavenet_model.add(tf.keras.layers.Dropout(0.1))

l2_reg = 0.001  # Define l2_reg (or pass it as an argument to the function)

for rate in [1, 2, 4, 8, 16, 32, 64, 128]:
    wavenet_model.add(tf.keras.layers.Conv1D(
        filters=32, kernel_size=2, padding="causal", activation=swish_activation,
        dilation_rate=rate,
        kernel_regularizer=tf.keras.regularizers.l2(l2_reg)  # Correct placement of regularizer
    ))
    wavenet_model.add(tf.keras.layers.Dropout(0.05))  # Moved dropout inside the loop

wavenet_model.add(tf.keras.layers.Conv1D(filters=4, kernel_size=1, kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))  # Regularization for final layer

# Softmax activation to convert logits to probabilities
wavenet_model.add(tf.keras.layers.Activation('softmax'))

# Compile the model
wavenet_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Define early stopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',  # Monitor the validation loss
    patience=1,          # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True  # Restore model weights from the epoch with the best value of the monitored quantity
)

# Train the model with early stopping
history = wavenet_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=150,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=[early_stopping, step_history]  # Add early stopping callback
)

# Evaluate the model
test_loss, test_accuracy = wavenet_model.evaluate(test_dataset, steps=test_steps)
print(f"Test accuracy: {test_accuracy}")

# Save the model and history
Wave_dir = os.path.join(output_dir, 'Wavenet_model_Test.keras')
model_save_path = Wave_dir
wavenet_model.save(model_save_path)
print(f"Model saved to {model_save_path}")

# Save the history to a JSON file
with open(os.path.join(output_dir, 'training_history.json'), 'w') as f:
    json.dump(history.history, f)

# Save the step history to a JSON file
with open(os.path.join(output_dir, 'step_history.json'), 'w') as f:
    json.dump({'step_losses': step_history.step_losses,
               'step_acc': step_history.step_acc,
               'val_step_losses': step_history.val_step_losses,
               'val_step_acc': step_history.val_step_acc}, f)

# Print the model summary
wavenet_model.summary()
# Print elapsed time
end_time = time.time()
elapsed_time = end_time - start_time
print("Elapsed time:", elapsed_time, "seconds")



NameError: name 'time' is not defined

In [ ]:
# Save the model
Wave_dir = os.path.join(output_dir, 'Model_Test.keras')
model_save_path = Wave_dir
wavenet_model.save(model_save_path)
print(f"Model saved to {model_save_path}")

In [ ]:
test_loss, test_accuracy = wavenet_model.evaluate(test_dataset, steps=test_steps)
print(f"Test accuracy: {test_accuracy}")

In [95]:
print(test_files)

['C:\\Users\\caspe\\OneDrive\\Documenten\\A_Casper\\Brein\\CS_Ai_Wolfhampton\\Deep_machine_learning\\CNN_EEG_project\\DataSetTest\\Output\\test_label_0.h5', 'C:\\Users\\caspe\\OneDrive\\Documenten\\A_Casper\\Brein\\CS_Ai_Wolfhampton\\Deep_machine_learning\\CNN_EEG_project\\DataSetTest\\Output\\test_label_1.h5', 'C:\\Users\\caspe\\OneDrive\\Documenten\\A_Casper\\Brein\\CS_Ai_Wolfhampton\\Deep_machine_learning\\CNN_EEG_project\\DataSetTest\\Output\\test_label_2.h5', 'C:\\Users\\caspe\\OneDrive\\Documenten\\A_Casper\\Brein\\CS_Ai_Wolfhampton\\Deep_machine_learning\\CNN_EEG_project\\DataSetTest\\Output\\test_label_3.h5']


In [73]:
import numpy as np
import h5py

def get_all_file_names(hdf5_file_path, labels):
    file_names = {label: set() for label in labels}
    with h5py.File(hdf5_file_path, 'r') as hdf5_file:
        for label in labels:
            label_group = hdf5_file[f'label_{label}']
            file_names[label] = set(label_group.keys())
    return file_names

def check_data_leakage(train_files, val_files, test_files):
    all_train_files = set().union(*train_files.values())
    all_val_files = set().union(*val_files.values())
    all_test_files = set().union(*test_files.values())

    # Check for overlaps between training and validation sets
    train_val_overlap = all_train_files.intersection(all_val_files)
    if train_val_overlap:
        print(f"Warning: {len(train_val_overlap)} files are in both training and validation sets.")
    else:
        print("No data leakage between training and validation sets.")

    # Check for overlaps between training and test sets
    train_test_overlap = all_train_files.intersection(all_test_files)
    if train_test_overlap:
        print(f"Warning: {len(train_test_overlap)} files are in both training and test sets.")
    else:
        print("No data leakage between training and test sets.")

    # Check for overlaps between validation and test sets
    val_test_overlap = all_val_files.intersection(all_test_files)
    if val_test_overlap:
        print(f"Warning: {len(val_test_overlap)} files are in both validation and test sets.")
    else:
        print("No data leakage between validation and test sets.")

# Define your file paths and labels
output_file = os.path.join(output_dir, 'dataset.h5')
labels = [0, 1, 2, 3]  # The labels in your dataset

# Get all file names for training, validation, and test sets
train_files = get_all_file_names(output_file, labels)  # Adjust if train/val/test split is stored differently
val_files = get_all_file_names(output_file, labels)    # Adjust if train/val/test split is stored differently
test_files = get_all_file_names(output_file, labels)   # Adjust if train/val/test split is stored differently

# Check for data leakage
check_data_leakage(train_files, val_files, test_files)


In [66]:
from sklearn.metrics import classification_report

# Assume the following function to get true labels and predictions from the test dataset
def get_true_labels_and_predictions(model, dataset, steps):
    true_labels = []
    predictions = []
    for step, (batch_data, batch_labels) in enumerate(dataset.take(steps)):
        preds = model.predict(batch_data)
        preds = np.argmax(preds, axis=-1).flatten()
        true_labels.extend(batch_labels.numpy().flatten())
        predictions.extend(preds)
    return np.array(true_labels), np.array(predictions)

# Get true labels and predictions from the test dataset
true_labels, predictions = get_true_labels_and_predictions(wavenet_model, test_dataset, test_steps)

# Generate classification report
report = classification_report(true_labels, predictions, output_dict=True)

# Extract the required metrics
metrics_table = {
    'Metric': ['Sensitivity (SEN)', 'Positive Predictive Value (PPV)', 'F1 Score'],
    'WaveNet Model': [
        report['weighted avg']['recall'],  # SEN
        report['weighted avg']['precision'],  # PPV
        report['weighted avg']['f1-score']  # F1
    ]
}

# Print the metrics table
metrics_df = pd.DataFrame(metrics_table)
print(metrics_df)


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
4/4 ━━━━━━━━

# Function to load data in batches from the HDF5 file
start_time = time.time()
dataset_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project'
output_dir = r'C:\Users\caspe\OneDrive\Documenten\A_Casper\Brein\CS_Ai_Wolfhampton\Deep_machine_learning\CNN_EEG_project\DataSetTest\Output'
#dir = os.path.join(dataset_dir, 'Dataset_Joined') #ataset Fnusa and Mayo together. Easier to get through the pipeline

dir = os.path.join(dataset_dir, 'DataSetTest') # Used to test pipeline 
# Define the output HDF5 file in order to minimize memory usage
output_file = os.path.join(output_dir, 'dataset.h5')
all_data = []
all_labels = []

def load_data_in_batches(hdf5_file, batch_size):
    for label in range(4):
        label_group = hdf5_file[f'label_{label}']
        file_names = list(label_group.keys())
        for i in range(0, len(file_names), batch_size):
            batch_files = file_names[i:i + batch_size]
            batch_data = []
            batch_labels = []
            for file_name in batch_files:
                data = label_group[file_name][:]
                batch_data.append(data)
                batch_labels.append(label)
            yield np.array(batch_data), np.array(batch_labels)

# Load and process data in batches
batch_size = 100  # Adjust the batch size as needed
with h5py.File(output_file, 'r') as hdf5_file:
    for batch_data, batch_labels in load_data_in_batches(hdf5_file, batch_size):
        all_data.append(batch_data)
        all_labels.append(batch_labels)
        end_time = time.time()
        elapsed_time = end_time - start_time
        print("Elapsed time:", elapsed_time, "seconds")

# Concatenate all the batches
all_data = np.concatenate(all_data, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

# Reshape labels if necessary
all_labels = all_labels[:, np.newaxis]

# Print shapes
print("All data shape:", all_data.shape)
print("All labels shape:", all_labels.shape)
end_time = time.time()
elapsed_time = end_time - start_time
print("Elapsed time:", elapsed_time, "seconds")


# Test instant upload

def load_and_split_data(file_path):
    all_data = []
    all_labels = []
    with h5py.File(file_path, 'r') as hdf5_file:
        for label in range(4):
            label_group = hdf5_file[f'label_{label}']
            for file_name in label_group.keys():
                data = label_group[file_name][:]
                all_data.append(data)
                all_labels.append(label)
    
    all_data = np.array(all_data)
    all_labels = np.array(all_labels)

    # Reshape data and labels if necessary
    all_data = all_data.reshape((all_data.shape[0], 1, 15000))
    all_labels = all_labels.reshape((all_labels.shape[0], 1))

    # First split: 60% training and 40% remaining
    train_data, remaining_data, train_labels, remaining_labels = train_test_split(
        all_data, all_labels, train_size=0.6, random_state=42, stratify=all_labels)

    # Second split: 50% of the remaining 40% goes to testing and 50% to validation
    test_data, val_data, test_labels, val_labels = train_test_split(
        remaining_data, remaining_labels, test_size=0.5, random_state=42, stratify=remaining_labels)

    return train_data, train_labels, val_data, val_labels, test_data, test_labels

# Function to create TensorFlow dataset from numpy arrays
def create_tf_dataset(data, labels, batch_size):
    tf_dataset = tf.data.Dataset.from_tensor_slices((data, labels))
    tf_dataset = tf_dataset.batch(batch_size)
    tf_dataset = tf_dataset.shuffle(buffer_size=10000)
    tf_dataset = tf_dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
    return tf_dataset

# Load and split data
train_data, train_labels, val_data, val_labels, test_data, test_labels = load_and_split_data(output_file)

# Define batch size
batch_size = 32  # Adjust as necessary

# Create TensorFlow datasets
train_dataset = create_tf_dataset(train_data, train_labels, batch_size)
val_dataset = create_tf_dataset(val_data, val_labels, batch_size)
test_dataset = create_tf_dataset(test_data, test_labels, batch_size)

# Verify the splits
print(f"Training data shape: {train_data.shape}")
print(f"Training labels shape: {train_labels.shape}")
print(f"Validation data shape: {val_data.shape}")
print(f"Validation labels shape: {val_labels.shape}")
print(f"Test data shape: {test_data.shape}")
print(f"Test labels shape: {test_labels.shape}")

#No BAtch

input_shape = (1, 15000)
wavenet_model = tf.keras.Sequential()
wavenet_model.add(tf.keras.layers.Input(shape=input_shape))

# Define Swish activation function
def swish_activation(x):
    return x * tf.sigmoid(x)

# Add dropout regularization
wavenet_model.add(tf.keras.layers.BatchNormalization())
wavenet_model.add(tf.keras.layers.Dropout(0.2))

for rate in [1, 2, 4, 8, 16, 32, 64, 128]:
    wavenet_model.add(tf.keras.layers.Conv1D(
        filters=32, kernel_size=2, padding="causal", activation=swish_activation,
        dilation_rate=rate))
    
    # Add dropout after each convolutional layer
    wavenet_model.add(tf.keras.layers.Dropout(0.2))

wavenet_model.add(tf.keras.layers.Conv1D(filters=14, kernel_size=1))

# Print the model summary
wavenet_model.summary()

In [27]:
# Compile the model
wavenet_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Define early stopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',  # Monitor the validation loss
    patience=8,          # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True  # Restore model weights from the epoch with the best value of the monitored quantity
)

# Train the model with early stopping
history = wavenet_model.fit(
    train_data, train_labels,
    epochs=150,
    validation_data=(val_data, val_labels),
    callbacks=[early_stopping]  # Add early stopping callback
)

# Evaluate the model
test_loss, test_accuracy = wavenet_model.evaluate(test_data, test_labels)
print(f"Test accuracy: {test_accuracy}")



Epoch 1/150
11/11 ━━━━━━━━━━━━━━━━━━━━ 4s 63ms/step - accuracy: 0.1710 - loss: 5.5205 - val_accuracy: 0.2174 - val_loss: 1.7672
Epoch 2/150
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.2607 - loss: 3.4161 - val_accuracy: 0.3565 - val_loss: 1.6164
Epoch 3/150
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.3120 - loss: 2.2384 - val_accuracy: 0.3565 - val_loss: 1.4964
Epoch 4/150
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.4074 - loss: 2.1548 - val_accuracy: 0.5478 - val_loss: 1.5952
Epoch 5/150
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.3521 - loss: 2.1030 - val_accuracy: 0.3565 - val_loss: 1.5952
Epoch 6/150
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.3235 - loss: 2.1909 - val_accuracy: 0.3565 - val_loss: 1.5574
Epoch 7/150
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.3362 - loss: 1.8575 - val_accuracy: 0.3565 - val_loss: 1.5567
Epoch 8/150
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.3968 - loss: 2.0209 - val_accuracy: 0.